In [14]:
import tskit
import pandas as pd
import sys
import os
import numpy as np
from collections import Counter
from math import ceil
from lets_plot import *
LetsPlot.setup_html()

foldType = "unfold"
popType = "pop"

removeSiblingsBoolean = True
removeMisclassBoolean = True

In [15]:

def multimap(s):
    if len(s.mutations) == 2:
        if not np.any([m.edge == tskit.NULL for m in s.mutations]):
            return s.id
    elif len(s.mutations) > 2:
        return s.id



In [17]:
# Read in meta
# %%
meta = pd.read_csv("/home/jobste/Data/TsinferMetaData_Vcf.txt")
meta.head()
allMeta = pd.read_csv("/home/jobste/Data/DataDescriptor_SampleMeta.csv")
allMeta.head()
meta.loc[:, "Country"] = [allMeta.Country[allMeta.Run == x].to_list()[0] for x in meta.ID]

samples = pd.read_csv("/home/jobste/Data/Remapped/NonHybrid_samples.txt", header = None, names = ["ID"])
sampleMeta = samples.merge(meta, how = "left", on = "ID")

sfsDF = pd.DataFrame()

for num,chromosome in enumerate([14, 15, 16]):
    # %%
    ts = tskit.load("/sietch_colab/jobste/Singer/Chr" + str(chromosome) + "/Chr" + str(chromosome) + "_NonHybrid_99.trees")
    remove_sites = [multimap(s) for s in ts.sites()]
    remove_sites = [x for x in remove_sites if x is not None]
    ts = ts.delete_sites(remove_sites)

    # %%
    # Add subpop as pop
    ######################################################
    if popType == "subpop":
        complex_schema_pop = tskit.MetadataSchema({
            'codec': 'json',
            'additionalProperties': False,
            'properties': {'pop': {'description': 'Lineage',
                                'type': 'string'},
                        'subpop': {'description': 'Subspecies',
                                'type': 'string'}},
            'required': ['pop'],
            'type': 'object',
        })

        tables = ts.dump_tables()
        tables.populations.metadata_schema = complex_schema_pop

        # Add data to population table
        popdata_column = [{"pop": pop, "subpop":subpop} for pop,subpop in set(list(zip(sampleMeta.Pop, sampleMeta.SubPop)))]
        encoded_popdata_column = [
            tables.populations.metadata_schema.validate_and_encode_row(r) for r in popdata_column
        ]
        md, md_offset = tskit.pack_bytes(encoded_popdata_column)
        tables.populations.set_columns(metadata=md, metadata_offset=md_offset)

        popDict = {y: x for x,y in enumerate(set(list(zip(sampleMeta.Pop, sampleMeta.SubPop))))}
        ind_nodes = list([x.id for x in ts.nodes() if x.time == 0.0])
        ind_pops = [popDict[(sampleMeta.Pop.to_list()[x], sampleMeta.SubPop.to_list()[x])] for x in ind_nodes]
    ######################################################3

    # Add lineage as pop
    ######################################################
    if popType == "pop":
        print(popType)
        complex_schema_pop = tskit.MetadataSchema({
            'codec': 'json',
            'additionalProperties': False,
            'properties': {'pop': {'description': 'Lineage',
                                'type': 'string'}},
            'required': ['pop'],
            'type': 'object',
        })

        tables = ts.dump_tables()
        tables.populations.metadata_schema = complex_schema_pop

        # Add data to population table
        popdata_column = [{"pop": pop} for pop in set(sampleMeta.Pop)]
        encoded_popdata_column = [
            tables.populations.metadata_schema.validate_and_encode_row(r) for r in popdata_column
        ]
        md, md_offset = tskit.pack_bytes(encoded_popdata_column)
        tables.populations.set_columns(metadata=md, metadata_offset=md_offset)

        popDict = {y: x for x,y in enumerate(set(sampleMeta.Pop))}
        ind_nodes = list([x.id for x in ts.nodes() if x.time == 0.0])
        ind_pops = [popDict[sampleMeta.Pop.to_list()[x]] for x in ind_nodes]
    ######################################################3


    indNodePop = tables.nodes.population
    indNodePop[ind_nodes] = ind_pops
    tables.nodes.population = indNodePop
    ts_meta = tables.tree_sequence()

    removeNames = pd.read_csv("/home/jobste/Singer/RemovedSibling.txt", names = ["ID"])
    keepNodes = [num for num,name in enumerate(sampleMeta.ID.to_list()) if name not in removeNames.ID.to_list()]
    ts_meta = ts_meta.simplify(keepNodes)

    removeNames = pd.read_csv("/home/jobste/Singer/RemovedSibling.txt", names = ["ID"])
    removeList = removeNames.ID.to_list()
    if removeSiblingsBoolean:
        removeSiblings = pd.read_csv("/home/jobste/Singer/IdentifiedSiblings.txt")
        removeList = removeList + removeSiblings.ID.to_list()
    if removeMisclassBoolean:
        removeMisclass = pd.read_csv("/home/jobste/Singer/IdentifiedMissclass.txt")
        removeList = removeList + removeMisclass.ID.to_list()
    keepNodes = [num for num,name in enumerate(sampleMeta.ID.to_list()) if name not in removeList]
    ts_meta = ts_meta.simplify(keepNodes)

    
    for popNum in range(ts_meta.num_populations):
        sample_set = list(ts_meta.samples(popNum))
        popName  = ts_meta.population(popNum).metadata[popType]
        print(popName)

        if foldType == "unfold":
            tmpSfs = ts_meta.allele_frequency_spectrum(sample_sets = [sample_set], polarised = True, span_normalise = False)


        elif foldType == "fold":
            tmpSfs = ts_meta.allele_frequency_spectrum(sample_sets = [sample_set], polarised = False, span_normalise = False)

        if foldType == "unfold":
            tmpSfsDF = pd.DataFrame({"Freq": tmpSfs[1:-1], "Count": range(1, len(tmpSfs[1:-1])+1)})
            tmpSfsDF.loc[:, "Pop"] = popName
            tmpSfsDF.loc[:, "Chr"] = chromosome
        elif foldType == "fold":
            tmpSfsDF = pd.DataFrame({"Freq": tmpSfs[1:ceil(len(sample_set)/2)], "Count": range(1, len(tmpSfs[1:ceil(len(sample_set)/2)])+1)})
            tmpSfsDF.loc[:, "Pop"] = popName
            tmpSfsDF.loc[:, "Chr"] = chromosome
        
        sfsDF = pd.concat([sfsDF, tmpSfsDF])


pop
A
M
O
C
pop
A
M
O
C
pop
A
M
O
C


In [18]:
sfsDF.loc[:, "LogFreq"] = np.log(sfsDF.Freq)

In [19]:
ggplot(data = sfsDF[sfsDF.Chr == 16]) + geom_point(aes(x = "Count", y = "LogFreq"))  + \
facet_wrap(facets = "Pop", scales ="free")

In [20]:
sfsDF_a = sfsDF.groupby(['Count', 'Pop'])['Freq'].sum().reset_index()

In [27]:
popType = "subpop"

# Read in meta
# %%
meta = pd.read_csv("/home/jobste/Data/TsinferMetaData_Vcf.txt")
meta.head()
allMeta = pd.read_csv("/home/jobste/Data/DataDescriptor_SampleMeta.csv")
allMeta.head()
meta.loc[:, "Country"] = [allMeta.Country[allMeta.Run == x].to_list()[0] for x in meta.ID]

samples = pd.read_csv("/home/jobste/Data/Remapped/NonHybrid_samples.txt", header = None, names = ["ID"])
sampleMeta = samples.merge(meta, how = "left", on = "ID")

sfsDF = pd.DataFrame()

for num,chromosome in enumerate([14, 15, 16]):
    # %%
    ts = tskit.load("/sietch_colab/jobste/Singer/Chr" + str(chromosome) + "/Chr" + str(chromosome) + "_NonHybrid_99.trees")
    remove_sites = [multimap(s) for s in ts.sites()]
    remove_sites = [x for x in remove_sites if x is not None]
    ts = ts.delete_sites(remove_sites)

    # %%
    # Add subpop as pop
    ######################################################
    if popType == "subpop":
        complex_schema_pop = tskit.MetadataSchema({
            'codec': 'json',
            'additionalProperties': False,
            'properties': {'pop': {'description': 'Lineage',
                                'type': 'string'},
                        'subpop': {'description': 'Subspecies',
                                'type': 'string'}},
            'required': ['pop'],
            'type': 'object',
        })

        tables = ts.dump_tables()
        tables.populations.metadata_schema = complex_schema_pop

        # Add data to population table
        popdata_column = [{"pop": pop, "subpop":subpop} for pop,subpop in set(list(zip(sampleMeta.Pop, sampleMeta.SubPop)))]
        encoded_popdata_column = [
            tables.populations.metadata_schema.validate_and_encode_row(r) for r in popdata_column
        ]
        md, md_offset = tskit.pack_bytes(encoded_popdata_column)
        tables.populations.set_columns(metadata=md, metadata_offset=md_offset)

        popDict = {y: x for x,y in enumerate(set(list(zip(sampleMeta.Pop, sampleMeta.SubPop))))}
        ind_nodes = list([x.id for x in ts.nodes() if x.time == 0.0])
        ind_pops = [popDict[(sampleMeta.Pop.to_list()[x], sampleMeta.SubPop.to_list()[x])] for x in ind_nodes]
    ######################################################3

    # Add lineage as pop
    ######################################################
    if popType == "pop":
        print(popType)
        complex_schema_pop = tskit.MetadataSchema({
            'codec': 'json',
            'additionalProperties': False,
            'properties': {'pop': {'description': 'Lineage',
                                'type': 'string'}},
            'required': ['pop'],
            'type': 'object',
        })

        tables = ts.dump_tables()
        tables.populations.metadata_schema = complex_schema_pop

        # Add data to population table
        popdata_column = [{"pop": pop} for pop in set(sampleMeta.Pop)]
        encoded_popdata_column = [
            tables.populations.metadata_schema.validate_and_encode_row(r) for r in popdata_column
        ]
        md, md_offset = tskit.pack_bytes(encoded_popdata_column)
        tables.populations.set_columns(metadata=md, metadata_offset=md_offset)

        popDict = {y: x for x,y in enumerate(set(sampleMeta.Pop))}
        ind_nodes = list([x.id for x in ts.nodes() if x.time == 0.0])
        ind_pops = [popDict[sampleMeta.Pop.to_list()[x]] for x in ind_nodes]
    ######################################################3


    indNodePop = tables.nodes.population
    indNodePop[ind_nodes] = ind_pops
    tables.nodes.population = indNodePop
    ts_meta = tables.tree_sequence()

    removeNames = pd.read_csv("/home/jobste/Singer/RemovedSibling.txt", names = ["ID"])
    keepNodes = [num for num,name in enumerate(sampleMeta.ID.to_list()) if name not in removeNames.ID.to_list()]
    ts_meta = ts_meta.simplify(keepNodes)

    removeNames = pd.read_csv("/home/jobste/Singer/RemovedSibling.txt", names = ["ID"])
    removeList = removeNames.ID.to_list()
    if removeSiblingsBoolean:
        removeSiblings = pd.read_csv("/home/jobste/Singer/IdentifiedSiblings.txt")
        removeList = removeList + removeSiblings.ID.to_list()
    if removeMisclassBoolean:
        removeMisclass = pd.read_csv("/home/jobste/Singer/IdentifiedMissclass.txt")
        removeList = removeList + removeMisclass.ID.to_list()
    keepNodes = [num for num,name in enumerate(sampleMeta.ID.to_list()) if name not in removeList]
    ts_meta = ts_meta.simplify(keepNodes)

    
    for popNum in range(ts_meta.num_populations):
        sample_set = list(ts_meta.samples(popNum))
        popName  = ts_meta.population(popNum).metadata[popType]
        print(popName)

        if foldType == "unfold":
            tmpSfs = ts_meta.allele_frequency_spectrum(sample_sets = [sample_set], polarised = True, span_normalise = False)


        elif foldType == "fold":
            tmpSfs = ts_meta.allele_frequency_spectrum(sample_sets = [sample_set], polarised = False, span_normalise = False)

        if len(tmpSfs) < 5:
            continue
        
        if foldType == "unfold":
            tmpSfsDF = pd.DataFrame({"Freq": tmpSfs[1:-1], "Count": range(1, len(tmpSfs[1:-1])+1)})
            tmpSfsDF.loc[:, "Pop"] = popName
            tmpSfsDF.loc[:, "Chr"] = chromosome
        elif foldType == "fold":
            tmpSfsDF = pd.DataFrame({"Freq": tmpSfs[1:ceil(len(sample_set)/2)], "Count": range(1, len(tmpSfs[1:ceil(len(sample_set)/2)])+1)})
            tmpSfsDF.loc[:, "Pop"] = popName
            tmpSfsDF.loc[:, "Chr"] = chromosome
        
        sfsDF = pd.concat([sfsDF, tmpSfsDF])


Buckfast
Scutellata
Ligustica
Carnica
Mellifera
Iberiensis
Caucasia
Unicolor
Buckfast
Scutellata
Ligustica
Carnica
Mellifera
Iberiensis
Caucasia
Unicolor
Buckfast
Scutellata
Ligustica
Carnica
Mellifera
Iberiensis
Caucasia
Unicolor


In [30]:
sfsDF.loc[:, "LogFreq"] = np.log(sfsDF.Freq)
ggplot(data = sfsDF[sfsDF.Chr == 16]) + geom_point(aes(x = "Count", y = "LogFreq"))  + \
facet_wrap(facets = "Pop", scales ="free")